# 🎓 Day 1 · Session 3
# 03B · Advanced Prompting Techniques
## Zero-shot, few-shot, step-by-step, self-reflection and prompt chaining

## 🎯 Learning Objectives

- Compare zero-shot and few-shot
- Use brief step-by-step prompts
- Apply self-reflection and prompt chaining

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from openai import OpenAI
import pandas as pd
import json

In [2]:
repo_root = Path.cwd()

while not (repo_root / ".git").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

env_path = repo_root / ".env"

print("Repository root:", repo_root)
print("Loading .env from:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(f"OPENAI_API_KEY not found in {env_path}")

client = OpenAI(api_key=api_key)

print("OpenAI client initialized successfully.")

Repository root: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai
Loading .env from: /Users/suresh/Projects/teaching/fdp-llm-agentic-ai/.env
Env exists: True
OpenAI client initialized successfully.


In [4]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.3,
               system_prompt="You are a helpful AI teaching assistant.",
               max_tokens=800):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ Zero-Shot Prompting

Zero-shot means no examples are provided.

In [5]:
zero_shot_prompt = """
Classify the sentiment of this student feedback as Positive, Negative or Mixed.

Feedback: The lab session was well organized, but the theory explanation was too fast.
"""
print(ask_openai(zero_shot_prompt))

The sentiment of the feedback is Mixed. The student expresses a positive sentiment about the organization of the lab session but also includes a negative aspect regarding the speed of the theory explanation.


# 2️⃣ Few-Shot Prompting

Few-shot prompting provides examples so the model imitates the pattern.

In [6]:
few_shot_prompt = """
Classify student feedback as Positive, Negative or Mixed.

Examples:
Feedback: The practical examples were excellent.
Label: Positive

Feedback: The class was confusing and too fast.
Label: Negative

Feedback: The content was useful, but the pace was too slow.
Label: Mixed

Now classify:
Feedback: The lab session was well organized, but the theory explanation was too fast.
Label:
"""
print(ask_openai(few_shot_prompt))

Mixed


# 3️⃣ Brief Step-by-Step Prompting

For teaching, ask for brief steps rather than hidden chain-of-thought.

In [7]:
problem = """
A train travels 300 km at 60 km/h, then 200 km at 100 km/h.
What is the average speed for the whole journey?
"""
print("DIRECT ANSWER")
print(ask_openai(problem + "\nGive only the answer."))

print("\nBRIEF STEPS")
print(ask_openai(problem + "\nShow brief calculation steps and then give the final answer."))

DIRECT ANSWER
The average speed for the whole journey is 75 km/h.

BRIEF STEPS
To find the average speed for the whole journey, we need to calculate the total distance traveled and the total time taken.

1. **Calculate total distance:**
   \[
   \text{Total distance} = 300 \text{ km} + 200 \text{ km} = 500 \text{ km}
   \]

2. **Calculate time for each segment:**
   - For the first segment (300 km at 60 km/h):
     \[
     \text{Time}_1 = \frac{300 \text{ km}}{60 \text{ km/h}} = 5 \text{ hours}
     \]
   - For the second segment (200 km at 100 km/h):
     \[
     \text{Time}_2 = \frac{200 \text{ km}}{100 \text{ km/h}} = 2 \text{ hours}
     \]

3. **Calculate total time:**
   \[
   \text{Total time} = 5 \text{ hours} + 2 \text{ hours} = 7 \text{ hours}
   \]

4. **Calculate average speed:**
   \[
   \text{Average speed} = \frac{\text{Total distance}}{\text{Total time}} = \frac{500 \text{ km}}{7 \text{ hours}} \approx 71.43 \text{ km/h}
   \]

**Final answer:**
The average speed for th

# 4️⃣ Self-Reflection Prompting

In [8]:
first_prompt = """
Create 5 MCQs on supervised learning for second-year engineering students.
Include answer key.
"""
initial_answer = ask_openai(first_prompt)
print(initial_answer)

reflection_prompt = f"""
Review the following MCQs.
Check for ambiguity, repeated options, incorrect answer key and difficulty balance.
Then provide an improved version.

MCQs:
{initial_answer}
"""
print("\nIMPROVED VERSION")
print(ask_openai(reflection_prompt, max_tokens=1200))

Sure! Here are five multiple-choice questions (MCQs) on supervised learning for second-year engineering students, along with the answer key.

### MCQs on Supervised Learning

**Question 1:**  
What is the primary goal of supervised learning?  
A) To cluster data points into groups  
B) To predict outcomes based on labeled input data  
C) To reduce the dimensionality of data  
D) To generate new data points  

**Question 2:**  
In supervised learning, what are the two main components of the dataset?  
A) Features and labels  
B) Clusters and centroids  
C) Inputs and outputs  
D) Training and testing  

**Question 3:**  
Which of the following algorithms is commonly used for classification tasks in supervised learning?  
A) K-Means  
B) Linear Regression  
C) Decision Trees  
D) Principal Component Analysis  

**Question 4:**  
What is overfitting in the context of supervised learning?  
A) When the model performs well on training data but poorly on unseen data  
B) When the model is to

# 5️⃣ Prompt Chaining

Split a complex task into smaller prompts.

In [9]:
topic = "decision trees"

explanation = ask_openai(f"Create a simple explanation of {topic} for second-year engineering students.")
mcqs = ask_openai(f"Based on this explanation, create 3 MCQs with answers.\n\n{explanation}")
activity = ask_openai(f"Create a 5-minute classroom activity based on this explanation and MCQs.\n\nExplanation:\n{explanation}\n\nMCQs:\n{mcqs}")

print("EXPLANATION\n", explanation)
print("\nMCQs\n", mcqs)
print("\nACTIVITY\n", activity)

EXPLANATION
 Sure! Let’s break down decision trees in a straightforward way.

### What is a Decision Tree?

A decision tree is a visual and analytical tool used for making decisions and predictions. It resembles a flowchart and helps you understand the possible outcomes of different choices.

### Structure of a Decision Tree

1. **Root Node**: This is the starting point of the tree, representing the entire dataset or the main decision to be made.

2. **Branches**: These are the lines that connect nodes, representing the different choices or outcomes that can arise from the decisions made at each node.

3. **Decision Nodes**: These are points in the tree where a decision is made based on a specific attribute (feature). Each decision node splits into branches based on the possible values of that attribute.

4. **Leaf Nodes**: These are the end points of the tree, representing the final outcomes or classifications resulting from the decisions made along the branches.

### How Does It Work

# 6️⃣ Step-Back Prompting

In [10]:
normal = "Why does overfitting happen in machine learning?"
step_back = """
Before answering, first identify the broader principle behind overfitting.
Then explain why overfitting happens in machine learning.
Keep the answer concise.
"""
print("NORMAL")
print(ask_openai(normal))
print("\nSTEP-BACK")
print(ask_openai(step_back))

NORMAL
Overfitting in machine learning occurs when a model learns not only the underlying patterns in the training data but also the noise and random fluctuations. This results in a model that performs well on the training data but poorly on unseen data (validation or test data). Here are some key reasons why overfitting happens:

1. **Complex Models**: When a model is too complex relative to the amount of training data, it can capture noise instead of just the signal. For example, deep neural networks with many layers and parameters can easily fit the training data perfectly.

2. **Insufficient Training Data**: If there is not enough training data, the model may learn specific details that do not generalize to new data. A small dataset can lead to a model that memorizes the training examples rather than learning general patterns.

3. **High Dimensionality**: In high-dimensional spaces, the volume of the space increases exponentially, making it easier for models to find spurious correl

# 🧪 Lab

Create zero-shot, few-shot, step-by-step, self-reflection and prompt-chain examples for your subject.

# ✅ Summary

Advanced prompting improves consistency, reasoning and usability, but faculty review is still mandatory.